In [1]:
import pandas as pd

df = pd.read_csv("data/thrillers.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6097 entries, 0 to 6096
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   original_title             6097 non-null   object 
 1   author                     6097 non-null   object 
 2   original_publication_year  6097 non-null   int64  
 3   num_pages                  5801 non-null   float64
 4   description                6028 non-null   object 
 5   genres                     6097 non-null   object 
 6   image_url                  6097 non-null   object 
 7   reviews_count              6097 non-null   int64  
 8   ratings_count              6097 non-null   int64  
 9   avg_rating                 6097 non-null   float64
dtypes: float64(2), int64(3), object(5)
memory usage: 476.5+ KB


In [ ]:
df = df[['original_title', 'description', 'ratings_count']].sort_values(by='ratings_count', ascending=False).head(200)
df = df.drop('ratings_count', axis=1)

,original_title,description,ratings_count
4296,The Hunger Games,Winning will make you famous.\nLosing means ce...,5066596
5406,Harry Potter and the Philosopher's Stone,Harry Potter's life is miserable. His parents ...,4972886
5875,To Kill a Mockingbird,The unforgettable novel of a childhood in a sl...,3402363
3166,Divergent,Paperback features over fifty pages of bonus m...,2277881
5275,Angels & Demons,An ancient secret brotherhood.\nA devastating ...,2126047
...,...,...,...
4204,From Dead to Worse,After the natural disaster of Hurricane Katrin...,162919
3595,The Hunger Games Box Set,"The extraordinary, ground breaking New York Ti...",162121
5328,Timeline,"In an Arizona desert, a man wanders in a daze,...",160870
4987,Darkly Dreaming Dexter,"Meet Dexter Morgan, a polite wolf in sheep's c...",160825


In [3]:
df = df.drop('ratings_count', axis=1)
df

,original_title,description
4296,The Hunger Games,Winning will make you famous.\nLosing means ce...
5406,Harry Potter and the Philosopher's Stone,Harry Potter's life is miserable. His parents ...
5875,To Kill a Mockingbird,The unforgettable novel of a childhood in a sl...
3166,Divergent,Paperback features over fifty pages of bonus m...
5275,Angels & Demons,An ancient secret brotherhood.\nA devastating ...
...,...,...
4204,From Dead to Worse,After the natural disaster of Hurricane Katrin...
3595,The Hunger Games Box Set,"The extraordinary, ground breaking New York Ti..."
5328,Timeline,"In an Arizona desert, a man wanders in a daze,..."
4987,Darkly Dreaming Dexter,"Meet Dexter Morgan, a polite wolf in sheep's c..."


In [4]:
df.to_csv('200thrillers.csv')

In [ ]:
import pandas as pd
from transformers import pipeline
import json
from pathlib import Path
import time

# ─────────────────────────────────────────────
# CONFIGURATION  — edit these before running
# ─────────────────────────────────────────────
DATA_PATH       = "200thrillers.csv"   # your 200-book sample
OUTPUT_PATH     = "mood_labels.csv"        # where results are saved
DESCRIPTION_COL = "description"           # column name in your CSV
TITLE_COL       = "original_title"        # column name in your CSV
THRESHOLD       = 0.60                    # minimum confidence to keep a label
BATCH_SIZE      = 10                       # books processed per batch

# Thriller-specific mood taxonomy
MOOD_LABELS = [
    "ominous",
    "sinister",
    "cerebral",
    "disorienting",
    "dark",
    "gore"
]


# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
def load_sample(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    # Drop books with no description — can't classify what we can't read
    missing = df[DESCRIPTION_COL].isna().sum()
    if missing > 0:
        print(f"  ⚠  Dropping {missing} books with missing descriptions")
    df = df.dropna(subset=[DESCRIPTION_COL]).reset_index(drop=True)

    print(f"  ✓  Loaded {len(df)} books for classification")
    return df


# ─────────────────────────────────────────────
# CLASSIFY
# ─────────────────────────────────────────────
def classify_moods(df: pd.DataFrame) -> pd.DataFrame:
    print("\nLoading model (first run downloads ~500MB)...")
    classifier = pipeline(
        task="zero-shot-classification",
        model="cross-encoder/nli-MiniLM2-L6-H768",
        device=-1,  # CPU — change to 0 if you have a GPU
    )
    print("  ✓  Model loaded\n")

    results = []
    total   = len(df)
    start   = time.time()

    for i, row in df.iterrows():
        description = str(row[DESCRIPTION_COL]).strip()
        title       = str(row.get(TITLE_COL, f"Book {i}"))

        # Run zero-shot classification
        # multi_label=True means labels are scored independently
        # (a book can be both dark AND suspenseful)
        output = classifier(
            description,
            candidate_labels=MOOD_LABELS,
            multi_label=True,
        )

        # Build score dict: {label: score}
        scores = dict(zip(output["labels"], output["scores"]))

        # Keep only labels above threshold
        moods_above_threshold = {
            label: round(score, 4)
            for label, score in scores.items()
            if score >= THRESHOLD
        }

        # Sort by confidence descending and keep top 3
        TOP_N = 3
        moods_sorted = dict(
            sorted(moods_above_threshold.items(), key=lambda x: x[1], reverse=True)[:TOP_N]
        )           

        results.append({
            TITLE_COL:      title,
            "moods":        json.dumps(list(moods_sorted.keys())),  # label list
            "mood_scores":  json.dumps(moods_sorted),               # full scores
            "mood_count":   len(moods_sorted),
        })

        # Progress + ETA
        elapsed     = time.time() - start
        avg_per_book = elapsed / (i + 1)
        remaining   = avg_per_book * (total - i - 1)
        print(
            f"  [{i+1:>3}/{total}]  {title[:40]:<40}"
            f"  moods: {list(moods_sorted.keys())}"
            f"  ETA: {remaining/60:.1f}min"
        )

    return pd.DataFrame(results)


# ─────────────────────────────────────────────
# DIAGNOSTICS  — quick sanity check on results
# ─────────────────────────────────────────────
def print_diagnostics(results_df: pd.DataFrame):
    print("\n" + "="*60)
    print("DIAGNOSTICS")
    print("="*60)

    # Mood coverage
    all_moods = []
    for moods_json in results_df["moods"]:
        all_moods.extend(json.loads(moods_json))

    from collections import Counter
    mood_counts = Counter(all_moods)

    print("\nMood frequency across all books:")
    for mood, count in mood_counts.most_common():
        bar   = "█" * int(count / len(results_df) * 20)
        pct   = count / len(results_df) * 100
        print(f"  {mood:<20} {bar:<20} {count:>3} books ({pct:.0f}%)")

    # Distribution of mood count per book
    print("\nMoods per book distribution:")
    dist = results_df["mood_count"].value_counts().sort_index()
    for n_moods, count in dist.items():
        print(f"  {n_moods} mood(s): {count} books")

    # Books with NO moods above threshold
    no_mood = results_df[results_df["mood_count"] == 0]
    if len(no_mood) > 0:
        print(f"\n  ⚠  {len(no_mood)} books got no mood above threshold {THRESHOLD}:")
        for _, row in no_mood.iterrows():
            print(f"     - {row[TITLE_COL]}")
        print(f"  → Consider lowering THRESHOLD from {THRESHOLD} to 0.15")
    else:
        print(f"\n  ✓  All books received at least one mood label")

    print("="*60)


# ─────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────
def save_results(results_df: pd.DataFrame, original_df: pd.DataFrame):
    # Merge mood results back onto original dataframe
    final_df = original_df.merge(results_df, on=TITLE_COL, how="left")
    final_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\n  ✓  Results saved to {OUTPUT_PATH}")
    print(f"     Columns added: moods, mood_scores, mood_count")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    print("="*60)
    print("ZERO-SHOT MOOD CLASSIFICATION")
    print(f"Model:     cross-encoder/nli-MiniLM2-L6-H768")
    print(f"Labels:    {MOOD_LABELS}")
    print(f"Threshold: {THRESHOLD}")
    print("="*60 + "\n")

    df         = load_sample(DATA_PATH)
    results_df = classify_moods(df)

    print_diagnostics(results_df)
    save_results(results_df, df)

    print("\nDone. Next steps:")
    print("  1. Open mood_labels.csv and spot-check 10-15 books you know well")
    print("  2. If too many books have 0 moods → lower THRESHOLD to 0.15")
    print("  3. If every book gets all 7 moods → raise THRESHOLD to 0.30")
    print("  4. Once happy with results, run on your full 6097-book dataset")


if __name__ == "__main__":
    main()

ZERO-SHOT MOOD CLASSIFICATION
Model:     cross-encoder/nli-MiniLM2-L6-H768
Labels:    ['dark', 'mysterious', 'suspenseful', 'fast-paced', 'mind-bending', 'emotional']
Threshold: 0.6

  ✓  Loaded 200 books for classification

Loading model (first run downloads ~500MB)...


Device set to use cpu


  ✓  Model loaded

  [  1/200]  The Hunger Games                          moods: ['emotional', 'suspenseful', 'fast-paced']  ETA: 5.6min
  [  2/200]  Harry Potter and the Philosopher's Stone  moods: ['mysterious', 'suspenseful', 'fast-paced']  ETA: 4.9min
  [  3/200]  To Kill a Mockingbird                     moods: ['emotional']  ETA: 4.1min
  [  4/200]  Divergent                                 moods: []  ETA: 4.3min
  [  5/200]  Angels & Demons                           moods: ['suspenseful']  ETA: 4.3min
  [  6/200]  Harry Potter and the Prisoner of Azkaban  moods: ['suspenseful', 'fast-paced', 'emotional']  ETA: 4.2min
  [  7/200]  Catching Fire                             moods: []  ETA: 4.1min
  [  8/200]  Man som hatar kvinnor                     moods: ['suspenseful', 'emotional', 'dark']  ETA: 4.0min
  [  9/200]  Harry Potter and the Chamber of Secrets   moods: ['mysterious', 'emotional', 'suspenseful']  ETA: 3.9min
  [ 10/200]  Mockingjay                                moods

PermissionError: [Errno 13] Permission denied: 'mood_labels.csv'